# 🧠 FaceKey Studio — Kaggle Dual T4 GPU Deep Learning Trainer
### Unleash 100% of Kaggle's Free 30h/Week GPU Power (2x NVIDIA T4 GPUs @ 500+ FPS)

> [!IMPORTANT]
> **CRITICAL KAGGLE SETTING: ACCELERATOR = GPU T4 x2**
> - In the right sidebar under **Notebook options**, select **GPU T4 x2**.
> - This allocates **2x NVIDIA Tesla T4 GPUs (30 GB combined VRAM, Tensor Cores)**.
> - Fully maximizes your free 30 hours/week GPU quota on PyTorch deep neural network training!

**Key Highlights:**
- **Dual GPU PyTorch `nn.DataParallel`**: Both T4 GPUs run at 100% load.
- **Automatic Mixed Precision (AMP FP16)**: Maximum Tensor Core throughput.
- **Zero Local Disk Wasted**: Automatically pulls the 13.5+ GB dataset chunks directly from Hugging Face Hub (`VijayTheOne/facekey-dataset-chunks`) over Kaggle's 1 Gbps datacenter network in ~90 seconds.
- **Interruption-Proof Cloud Sync**: Auto-saves and syncs model checkpoints (`checkpoint_best.pt`, `checkpoint_latest.pt`) directly back to Hugging Face Hub every epoch!

## ⚡ STEP 1: VERIFY DUAL NVIDIA T4 GPUs & TENSOR CORES

In [ ]:
# Verify Dual T4 GPU allocation
!nvidia-smi

import torch
gpu_count = torch.cuda.device_count()
print(f'\n[+] PyTorch CUDA Available: {torch.cuda.is_available()}')
print(f'[+] Detected GPUs: {gpu_count}')
for i in range(gpu_count):
    print(f'    • GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / (1024**3):.1f} GB VRAM)')
if gpu_count >= 2:
    print('[✓] DUAL T4 GPU ACCELERATION READY!')
elif gpu_count == 1:
    print('[!] Single GPU detected. (For maximum speed, switch Accelerator to GPU T4 x2 in Notebook options).')
else:
    print('[!] WARNING: No GPU detected! Set Accelerator to GPU T4 x2 in the right sidebar.')


## ⚡ STEP 2: CLONE & UPDATE FACEKEY REPOSITORY (SELF-HEALING)

In [ ]:
# Clone or pull latest repository code
import os
%cd /kaggle/working

APP_DIR = '/kaggle/working/FaceKeyAnimation'
if os.path.isdir(os.path.join(APP_DIR, '.git')):
    print('[*] Updating FaceKeyAnimation to latest GitHub main...')
    !cd {APP_DIR} && git fetch --all --prune && git reset --hard origin/main && git clean -fd
else:
    print('[*] Fresh clone of FaceKeyAnimation from GitHub...')
    !rm -rf {APP_DIR}
    !git clone https://github.com/Vijay-1010110/FaceKeyAnimation.git {APP_DIR}

%cd {APP_DIR}
!pip install -q -U torch torchvision torchaudio huggingface_hub numpy scipy
print('[+] Deep Learning Training environment ready!')


## ⚡ STEP 3: LAUNCH DUAL T4 GPU TRAINING (AUTO-DOWNLOADS CHUNKS & AUTO-SAVES WEIGHTS)

In [ ]:
# ==============================================================================
# 🚀 1-CLICK DUAL T4 GPU MODEL TRAINING (SPEECH-TO-FACIAL ANIMATION)
# ==============================================================================
import os, sys

# 1. Resolve Hugging Face Credentials
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN and os.path.exists('/kaggle/working/hf_token.txt'):
    with open('/kaggle/working/hf_token.txt') as f:
        HF_TOKEN = f.read().strip()
if not HF_TOKEN:
    HF_TOKEN = input('Enter Hugging Face Token: ').strip()
    if HF_TOKEN:
        with open('/kaggle/working/hf_token.txt', 'w') as f:
            f.write(HF_TOKEN)

HF_REPO = 'VijayTheOne/facekey-dataset-chunks'

# 2. Launch Multi-GPU Training Engine
# - Automatically pulls chunks from Hugging Face if training_data is empty
# - Runs DataParallel across Dual T4 GPUs with FP16 Tensor Cores
# - Saves best and latest checkpoints to Hugging Face Hub after every epoch!
%cd /kaggle/working/FaceKeyAnimation
!python scripts/train_speech_to_animation.py \
    --data-dir '/kaggle/working/training_data' \
    --checkpoints-dir '/kaggle/working/checkpoints' \
    --hf-repo '{HF_REPO}' \
    --hf-token '{HF_TOKEN}' \
    --epochs 50 \
    --batch-size 128 \
    --lr 1e-4 \
    --seq-len 64


---
## 🚨 STEP 4: EMERGENCY RESCUE & SAVE CHECKPOINTS TO HUGGING FACE
> Run this cell anytime if training is stopped or before your Kaggle session finishes to ensure all saved weights are in your Hugging Face cloud repository.

In [ ]:
# Force sync checkpoints to Hugging Face Hub
import os, glob
from huggingface_hub import HfApi

ckpts = glob.glob('/kaggle/working/checkpoints/*.pt')
if not ckpts:
    print('[!] No .pt checkpoints found in /kaggle/working/checkpoints')
else:
    api = HfApi()
    print(f'[*] Uploading {len(ckpts)} checkpoint(s) to Hugging Face...')
    for ck in ckpts:
        fname = os.path.basename(ck)
        print(f'  -> Uploading {fname}...', end=' ', flush=True)
        api.upload_file(
            path_or_fileobj=ck,
            path_in_repo=f'checkpoints/{fname}',
            repo_id=HF_REPO,
            repo_type='dataset',
            token=HF_TOKEN
        )
        print('[DONE]')
    print('[✓] All checkpoints safely secured in Hugging Face Hub!')
